# 🥚 Egg Yolk Color Prediction - Model Training & Evaluation Notebook

สมุดบันทึกประเมินและเปรียบเทียบประสิทธิภาพของโมเดล Machine Learning 5 ตัว
ด้วยวิธี **Stratified 5-Fold Cross-Validation** (รักษาสัดส่วนคลาส 80/20)
บนชุดข้อมูลฟีเจอร์สีที่ผ่านการสกัดด้วยเทคนิค **Center Circular Masking (R=42%)** เพื่อตัดขอบพื้นหลังโต๊ะไม้ออก 100%

คำนวณและแสดงผลตัววัดผลทางสถิติของ Regression ล้วนๆ ได้แก่ **Train R², Test R², Train MAE, Test MAE, Train RMSE, Test RMSE** และ **Per-Class Regression Error Breakdown (Mean Pred, MAE, RMSE รายคลาส)**

In [1]:
# 1. Import Libraries
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
# 2. Load Features & Stratified 5-Fold Cross-Validation Evaluation (R2, MAE, RMSE)
features_csv = 'data/features.csv'
df = pd.read_csv(features_csv)
print(f"Loaded features dataset: {len(df)} samples across 12 classes.\n")

feature_cols = ['r', 'g', 'b', 'l', 'a', 'b_lab']
X = df[feature_cols].values
y = df['fan_score'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'SVR (RBF Kernel)': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', SVR(kernel='rbf', C=10.0, epsilon=0.1))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=1.0))
    ])
}

comparison_results = []
per_class_results = []
classes = sorted(np.unique(y))

for name, pipeline in models.items():
    tr_r2, te_r2 = [], []
    tr_mae, te_mae = [], []
    tr_rmse, te_rmse = [], []

    all_true, all_pred = [], []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[val_idx], y[val_idx]

        pipeline.fit(X_tr, y_tr)

        # Predict Train
        p_tr = pipeline.predict(X_tr)
        tr_r2.append(r2_score(y_tr, p_tr))
        tr_mae.append(mean_absolute_error(y_tr, p_tr))
        tr_rmse.append(np.sqrt(mean_squared_error(y_tr, p_tr)))

        # Predict Test (Validation)
        p_te = pipeline.predict(X_te)
        te_r2.append(r2_score(y_te, p_te))
        te_mae.append(mean_absolute_error(y_te, p_te))
        te_rmse.append(np.sqrt(mean_squared_error(y_te, p_te)))

        all_true.extend(y_te)
        all_pred.extend(p_te)

    comparison_results.append({
        'Model': name,
        'Train R2': round(np.mean(tr_r2), 4),
        'Test R2': round(np.mean(te_r2), 4),
        'Train MAE': round(np.mean(tr_mae), 4),
        'Test MAE': round(np.mean(te_mae), 4),
        'Train RMSE': round(np.mean(tr_rmse), 4),
        'Test RMSE': round(np.mean(te_rmse), 4)
    })

    # Calculate per-class regression error metrics on Test set
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    for c in classes:
        mask = (all_true == c)
        preds = all_pred[mask]
        mae_c = mean_absolute_error(all_true[mask], preds)
        rmse_c = np.sqrt(mean_squared_error(all_true[mask], preds))
        
        per_class_results.append({
            'Model': name,
            'Class (Fan Score)': c,
            'Samples': int(np.sum(mask)),
            'Mean Pred': round(np.mean(preds), 2),
            'MAE': round(mae_c, 4),
            'RMSE': round(rmse_c, 4)
        })

res_df = pd.DataFrame(comparison_results).sort_values(by='Test R2', ascending=False)
per_class_df = pd.DataFrame(per_class_results)

print('=' * 90)
print('MODEL TRAIN vs TEST COMPARISON RESULTS (Stratified 5-Fold Cross-Validation)')
print('=' * 90)
print(res_df.to_string(index=False))
print('=' * 90)
best_model_name = res_df.iloc[0]['Model']
print(f"\nBest Performing Model: {best_model_name} (Test R^2 = {res_df.iloc[0]['Test R2']:.4f})")

Loaded features dataset: 647 samples across 12 classes.

MODEL TRAIN vs TEST COMPARISON RESULTS (Stratified 5-Fold Cross-Validation)
            Model  Train R2  Test R2  Train MAE  Test MAE  Train RMSE  Test RMSE
 SVR (RBF Kernel)    0.9295   0.9175     0.5814    0.6453      0.7915     0.8532
    Random Forest    0.9871   0.9087     0.2462    0.6485      0.3389     0.8923
Gradient Boosting    0.9663   0.9063     0.4209    0.6621      0.5473     0.9038
Linear Regression    0.8749   0.8718     0.8074    0.8136      1.0542     1.0645
 Ridge Regression    0.8574   0.8545     0.8775    0.8835      1.1256     1.1355

Best Performing Model: SVR (RBF Kernel) (Test R^2 = 0.9175)


In [3]:
# 3. Display Per-Class Regression Error Breakdown for Top Model (SVR)
top_model_name = res_df.iloc[0]['Model']
top_per_class = per_class_df[per_class_df['Model'] == top_model_name]

print(f"=== PER-CLASS REGRESSION ERROR BREAKDOWN FOR TOP MODEL ({top_model_name}) ===")
print(top_per_class.to_string(index=False))
print('=' * 90)

=== PER-CLASS REGRESSION ERROR BREAKDOWN FOR TOP MODEL (SVR (RBF Kernel)) ===
           Model  Class (Fan Score)  Samples  Mean Pred    MAE   RMSE
SVR (RBF Kernel)                  4       36       4.29 0.3437 0.6434
SVR (RBF Kernel)                  5       41       5.48 0.7397 0.9146
SVR (RBF Kernel)                  6       49       6.14 0.6864 0.8799
SVR (RBF Kernel)                  7       42       7.29 0.7035 0.9421
SVR (RBF Kernel)                  8       64       8.26 0.5288 0.6512
SVR (RBF Kernel)                  9      105       9.06 0.6079 0.7664
SVR (RBF Kernel)                 10       99      10.02 0.6547 0.8258
SVR (RBF Kernel)                 11       75      10.77 0.6839 0.9368
SVR (RBF Kernel)                 12       33      11.68 0.8708 1.1319
SVR (RBF Kernel)                 13       21      12.66 0.9676 1.0728
SVR (RBF Kernel)                 14       33      13.47 0.7329 0.9236
SVR (RBF Kernel)                 15       49      14.61 0.5019 0.8145
